# Part 2: Memory Management & Advanced NumPy
In this notebook, we cover the most common reasons machine learning pipelines crash local machines: memory leaks and silent data corruption. 

**Topics Covered:**
1. **The Silent Bug:** Views vs. Copies (How physical RAM works)
2. **Memory Optimization:** In-Place Operations (Avoiding the RAM Spike)
3. **Advanced Indexing:** High-speed data filtering with Boolean Masking

In [1]:
import numpy as np
print("NumPy imported successfully!")

NumPy imported successfully!


---
### 1. The Silent Bug: Views vs. Copies
When you slice a NumPy array (e.g., `data[0:2]`), it **does not** create a new array. It creates a "View". 

Think of a View like a second security camera looking at the exact same physical block of RAM. If you alter the view, you permanently mutate the original data. Python will not throw an error, which can silently corrupt your entire AI dataset.

In [2]:
# Create an array of base weights for our model
weights_a = np.array([1.0, 2.0, 3.0, 4.0])

# We slice the array to experiment with the first two weights
weights_b = weights_a[0:2] 

# Change the first value of our new slice to 99
weights_b[0] = 99.0

# Watch what happens to the ORIGINAL data
print("New Weights (b):", weights_b)
print("Original Weights (a):", weights_a) 

# Verify they share the exact same physical memory block
print("\nDo they share memory?", np.shares_memory(weights_a, weights_b))

New Weights (b): [99.  2.]
Original Weights (a): [99.  2.  3.  4.]

Do they share memory? True


### The Fix: Forcing a Copy
To isolate data (like when separating your Training data from your Testing data), you must explicitly allocate new memory using the `.copy()` command. This forces the CPU to carve out a brand new, safe block of RAM.

In [3]:
# Force the CPU to allocate new RAM
weights_c = weights_a[0:2].copy()

# Modify the copy
weights_c[0] = 55.0

print("Original Weights (a):", weights_a)
print("Copied Weights (c):", weights_c)
print("\nDo they share memory?", np.shares_memory(weights_a, weights_c))

Original Weights (a): [99.  2.  3.  4.]
Copied Weights (c): [55.  2.]

Do they share memory? False


---
### 2. In-Place Operations: Preventing the "RAM Spike"
If you are training models locally on a machine with 16GB or 32GB of RAM, standard math can crash your system. 

Running `x = x * 2` creates a massive temporary array in memory to hold the answer before replacing the old variable. This causes a massive "RAM Spike". Instead, use **In-Place Operators** like `*=` to overwrite the original memory block directly with zero overhead.

In [4]:
# Create a dummy dataset 
massive_data = np.ones((5, 5)) 

# The memory-efficient way to multiply: Overwrite the existing RAM block directly
massive_data *= 2

print("In-Place Multiplication Result:\n", massive_data)

In-Place Multiplication Result:
 [[2. 2. 2. 2. 2.]
 [2. 2. 2. 2. 2.]
 [2. 2. 2. 2. 2.]
 [2. 2. 2. 2. 2.]
 [2. 2. 2. 2. 2.]]


---
### 3. Boolean Masking: The Hardware Stencil
Writing standard `for` loops with `if/else` statements to filter millions of data points is too slow for machine learning. Instead, we use a two-step vectorized process called Boolean Masking:

1. **Create the Mask:** We write a condition that generates an array of `True/False` values.
2. **Apply the Mask:** We pass that array back into the brackets. NumPy acts like a physical stencil, ignoring the `False` values and extracting only the `True` data.

In [5]:
pixels = np.array([10, 255, 128, 5, 200, 50])

# Step 1: Create a mask for pixels brighter than 100
mask = pixels > 100
print("The Boolean Mask:", mask)

# Step 2: Apply the mask to extract data
bright_pixels = pixels[mask]
print("\nFiltered Bright Pixels:", bright_pixels)

# Pro-Move: We can zero out all dark pixels in a single line of code
pixels[pixels < 100] = 0
print("\nImage array with dark pixels removed:", pixels)

The Boolean Mask: [False  True  True False  True False]

Filtered Bright Pixels: [255 128 200]

Image array with dark pixels removed: [  0 255 128   0 200   0]
